# H&M 数据探索与核对 / H&M Data Exploration & Validation

**这个 notebook 做什么 / What this notebook does**

- 记录数据探索的过程：每个口径决定（批量阈值、价格缩放、删失截止日期等）背后的查询和证据。
  Records the exploration behind every definition decision (bulk threshold, price scaling, censoring cut-offs, etc.).
- 不建表。所有建表 SQL 在 `sql/` 中，由 `build_db.py` 执行；这里只读查询。
  It does not build tables. All table SQL lives in `sql/` and is run by `build_db.py`; this notebook only reads.
- 口径与结论的正式记录见 `docs/data_dictionary.md`，各节标注了对应章节。
  The formal record of definitions is `docs/data_dictionary.md`; each section below points to the matching part.

**运行顺序 / Run order**

```bash
python build_db.py   # 从原始 CSV 建库 / build the database from raw CSVs
python checks.py     # 跨层核对 / cross-layer checks
```
然后再打开本 notebook。/ Then open this notebook.

> ⚠️ DuckDB 同一时间只允许一个进程写入。运行 `build_db.py` 前，请先关闭本 notebook 的 kernel。
> DuckDB allows only one writing process at a time. Shut down this notebook's kernel before running `build_db.py`.

## 0. 环境设置 / Setup


In [4]:
from pathlib import Path
import duckdb


def find_project_root(marker=".git"):
    """从当前目录向上找，第一个包含 marker 的文件夹就是项目根目录。
    Walk up from the current folder; the first one containing marker is the root."""
    here = Path.cwd()
    for folder in [here, *here.parents]:
        if (folder / marker).exists():
            return folder
    raise FileNotFoundError(f"No {marker} found above {here}")


ROOT = find_project_root()
con = duckdb.connect(str(ROOT / "data" / "hm.duckdb"), read_only=True)
print(ROOT)

/Users/valerieyin/GitRepo/hm-analysis


## 1. ODS 摸底 / First look at ODS
对应数据字典 §1–§3 / See data dictionary §1–§3

**问题 / Questions**：每张表多大？时间范围？价格是什么尺度？渠道怎么分布？
How big is each table? What is the date range? What scale is price on? How are channels split?

In [5]:
# 每张表的行数 / Row count per table
for t in ["ods_transactions", "ods_articles", "ods_customers"]:
    print(t, con.sql(f"SELECT COUNT(*) FROM {t}").fetchone()[0])

ods_transactions 31788324
ods_articles 105542
ods_customers 1371980


In [6]:
# 交易表概况：日期范围、去重顾客 / 商品数、价格分布
# Transaction overview: date range, distinct customers / articles, price distribution
con.sql("""
SELECT
    MIN(t_dat)                  AS first_day,
    MAX(t_dat)                  AS last_day,
    COUNT(DISTINCT customer_id) AS n_customers,
    COUNT(DISTINCT article_id)  AS n_articles,
    MIN(price)                  AS min_price,
    QUANTILE_CONT(price, 0.5)   AS median_price,
    MAX(price)                  AS max_price
FROM ods_transactions
""").show()

┌────────────┬────────────┬─────────────┬────────────┬───────────────────────┬──────────────────────┬────────────────────┐
│ first_day  │  last_day  │ n_customers │ n_articles │       min_price       │     median_price     │     max_price      │
│    date    │    date    │    int64    │   int64    │        double         │        double        │       double       │
├────────────┼────────────┼─────────────┼────────────┼───────────────────────┼──────────────────────┼────────────────────┤
│ 2018-09-20 │ 2020-09-22 │     1362281 │     104547 │ 1.694915254237288e-05 │ 0.025406779661016947 │ 0.5915254237288136 │
└────────────┴────────────┴─────────────┴────────────┴───────────────────────┴──────────────────────┴────────────────────┘



In [7]:
# 渠道占比：SUM(COUNT(*)) OVER () 是窗口函数，把总数附到每一行上作为分母
# Channel share: SUM(COUNT(*)) OVER () is a window function that attaches the grand total to every row
con.sql("""
SELECT sales_channel_id,
       COUNT(*) AS n,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM ods_transactions
GROUP BY sales_channel_id
""").show()

┌──────────────────┬──────────┬────────┐
│ sales_channel_id │    n     │  pct   │
│       int8       │  int64   │ double │
├──────────────────┼──────────┼────────┤
│                1 │  9408462 │   29.6 │
│                2 │ 22379862 │   70.4 │
└──────────────────┴──────────┴────────┘



**全量数据结果 / Results on the full data**

- 交易 31,788,324 行；商品表 105,542 个，其中 104,547 个有销售；顾客表 1,371,980 人，其中 1,362,281 人有购买。
  31,788,324 transactions; 104,547 of 105,542 articles sold; 1,362,281 of 1,371,980 customers purchased.
- 日期 2018-09-20 至 2020-09-22，首尾两周不完整；2020 年包含疫情期间。
  Dates run 2018-09-20 to 2020-09-22; the first and last weeks are partial; 2020 includes the COVID period.
- 价格最大值只有 0.59，明显经过缩放，不是货币金额。
  Max price is only 0.59, so price is scaled and not a currency amount.
- 渠道 1 占 29.6%，渠道 2 占 70.4%；官方未说明哪个是线上，报告中只写「渠道 1 / 渠道 2」。
  Channel 1 is 29.6%, channel 2 is 70.4%; which is online is undocumented, so reports say "channel 1 / channel 2" only.

In [8]:
# 列类型：注意商品表的 article_id 被自动识别成 VARCHAR
# Column types: note that article_id in the articles table is auto-detected as VARCHAR
print(con.sql("DESCRIBE ods_articles").df().to_string())
print(con.sql("DESCRIBE ods_customers").df().to_string())

                     column_name column_type null   key default extra
0                     article_id     VARCHAR  YES  None    None  None
1                   product_code     VARCHAR  YES  None    None  None
2                      prod_name     VARCHAR  YES  None    None  None
3                product_type_no      BIGINT  YES  None    None  None
4              product_type_name     VARCHAR  YES  None    None  None
5             product_group_name     VARCHAR  YES  None    None  None
6        graphical_appearance_no      BIGINT  YES  None    None  None
7      graphical_appearance_name     VARCHAR  YES  None    None  None
8              colour_group_code     VARCHAR  YES  None    None  None
9              colour_group_name     VARCHAR  YES  None    None  None
10     perceived_colour_value_id      BIGINT  YES  None    None  None
11   perceived_colour_value_name     VARCHAR  YES  None    None  None
12    perceived_colour_master_id      BIGINT  YES  None    None  None
13  perceived_colour

In [9]:
# 卖出的商品是否都能在商品表中找到？
# LEFT JOIN 保留左边每一行；匹配不上时 a.article_id 为 NULL，COUNT(列) 不计 NULL
# Can every sold article be found in the articles table?
# LEFT JOIN keeps every left row; unmatched rows get NULL, and COUNT(column) skips NULL
con.sql("""
SELECT COUNT(*)            AS sold_articles,
       COUNT(a.article_id) AS matched
FROM (SELECT DISTINCT article_id FROM ods_transactions) t
LEFT JOIN ods_articles a
       ON t.article_id = CAST(a.article_id AS BIGINT)
""").show()

┌───────────────┬─────────┐
│ sold_articles │ matched │
│     int64     │  int64  │
├───────────────┼─────────┤
│        104547 │  104547 │
└───────────────┴─────────┘



**结果 / Result**：全量数据 104,547 个卖出商品 100% 匹配 → DWD 层统一 `CAST(article_id AS BIGINT)`。
All 104,547 sold articles match → DWD casts `article_id` to BIGINT.

## 2. 价格缩放 / Price scaling
对应数据字典 §4.2 / See data dictionary §4.2

**假设 / Hypothesis**：最小值 = 1/59000，中位数 × 59 = 1.499，猜测 price × 59 会得到以 99 结尾的价格点。
The minimum is 1/59000 and median × 59 = 1.499, suggesting price × 59 yields price points ending in 99.

In [10]:
# 随机抽 10 行验证；ROUND 消除浮点误差（如 1.9989999999999997 → 1.999）
# Sample 10 rows to verify; ROUND removes float noise (e.g. 1.9989999999999997 → 1.999)
con.sql("""
SELECT price, ROUND(price * 59, 3) AS price_point
FROM ods_transactions
USING SAMPLE 10
""").show()

┌──────────────────────┬─────────────┐
│        price         │ price_point │
│        double        │   double    │
├──────────────────────┼─────────────┤
│  0.05930508474576271 │       3.499 │
│ 0.005067796610169492 │       0.299 │
│  0.01083050847457627 │       0.639 │
│ 0.033881355932203386 │       1.999 │
│ 0.033881355932203386 │       1.999 │
│  0.03049152542372881 │       1.799 │
│  0.08133898305084745 │       4.799 │
│ 0.005915254237288136 │       0.349 │
│ 0.033881355932203386 │       1.999 │
│  0.00947457627118644 │       0.559 │
└──────────────────────┴─────────────┘
  10 rows                  2 columns



**结论 / Conclusion**：价格点呈 x.x99 规律，但这只是推测，未经官方确认。
**决定**：只用相对价格、价格带和变化率，不报告货币金额；DWD 层保存 `price_point = ROUND(price * 59, 3)`。
Price points follow the x.x99 pattern, but this is inferred, not officially confirmed.
**Decision**: use only relative prices, price bands and percentage changes; DWD stores `price_point = ROUND(price * 59, 3)`.

## 3. 同日同商品多行与批量购买 / Repeated rows and bulk purchases
对应数据字典 §4.1、§4.3 / See data dictionary §4.1 and §4.3

**问题 / Question**：同一顾客同一天同一商品出现多行，是重复数据还是买了多件？
When the same customer buys the same article on the same day in several rows, is that duplication or several units?

**判断依据 / Reasoning**：交易表没有数量列，一行只能代表一件；`article_id` 不含尺码，买两个尺码就是同一商品两行。Kaggle 官方说明也确认重复行代表多次购买。
There is no quantity column, so one row is one unit; `article_id` has no size, so two sizes show up as two rows of one article. Kaggle's data page confirms duplicate rows are multiple purchases.

In [11]:
# 每个（日期，顾客，商品）组合有几行，统计各行数档的组合个数
# GROUP BY ALL = 按所有非聚合列分组
# Rows per (date, customer, article) group, then how many groups fall in each size
# GROUP BY ALL = group by every non-aggregate column
con.sql("""
WITH g AS (
    SELECT t_dat, customer_id, article_id, COUNT(*) AS n_rows
    FROM ods_transactions
    GROUP BY ALL
)
SELECT n_rows, COUNT(*) AS n_groups
FROM g
GROUP BY n_rows
ORDER BY n_rows
LIMIT 10
""").show()

┌────────┬──────────┐
│ n_rows │ n_groups │
│ int64  │  int64   │
├────────┼──────────┤
│      1 │ 25857551 │
│      2 │  2418787 │
│      3 │   205350 │
│      4 │    63677 │
│      5 │    10207 │
│      6 │    10013 │
│      7 │     1984 │
│      8 │     2429 │
│      9 │      967 │
│     10 │     1300 │
└────────┴──────────┘
       10 rows     



In [12]:
# 多行组合里价格或渠道是否不同？不同的组合不可能是机械复制
# SUM(CASE WHEN ...) 是条件计数
# Do multi-row groups differ in price or channel? Differing groups cannot be mechanical copies
# SUM(CASE WHEN ...) is conditional counting
con.sql("""
WITH g AS (
    SELECT t_dat, customer_id, article_id,
           COUNT(*)                         AS n_rows,
           COUNT(DISTINCT price)            AS n_prices,
           COUNT(DISTINCT sales_channel_id) AS n_channels
    FROM ods_transactions
    GROUP BY ALL
)
SELECT
    COUNT(*)                                        AS multi_row_groups,
    SUM(CASE WHEN n_prices > 1 THEN 1 ELSE 0 END)   AS diff_price,
    SUM(CASE WHEN n_channels > 1 THEN 1 ELSE 0 END) AS diff_channel
FROM g
WHERE n_rows > 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬────────────┬──────────────┐
│ multi_row_groups │ diff_price │ diff_channel │
│      int64       │   int128   │    int128    │
├──────────────────┼────────────┼──────────────┤
│          2717844 │     227586 │         8494 │
└──────────────────┴────────────┴──────────────┘



In [13]:
# 行数最多的组合：远超个人消费的就是批量购买
# Largest groups: far beyond personal shopping means bulk buying
con.sql("""
SELECT t_dat, customer_id, article_id, COUNT(*) AS n_rows
FROM ods_transactions
GROUP BY ALL
ORDER BY n_rows DESC
LIMIT 5
""").show()

┌────────────┬──────────────────────────────────────────────────────────────────┬────────────┬────────┐
│   t_dat    │                           customer_id                            │ article_id │ n_rows │
│    date    │                             varchar                              │   int64    │ int64  │
├────────────┼──────────────────────────────────────────────────────────────────┼────────────┼────────┤
│ 2018-10-14 │ d00063b94dcb1342869d4994844a2742b5d62927f36843164fb3f818f630bca9 │  678342001 │    570 │
│ 2019-02-16 │ 94665b46e194622ccdbcadc0170f13a2f8ede1ff6d057d43a19b8938c808b662 │  629420001 │    199 │
│ 2019-04-15 │ ebc0f83524828ee05e9dc70b50d21a261000d2015f6b2006289199a93ccf0702 │  685347001 │    130 │
│ 2019-09-28 │ 5cba04ed9a3759bc02a8a9e01efccc07ce76c35c1a70dce6c5cedba32ce3ab6b │  688558002 │    121 │
│ 2019-02-11 │ 8f5f1e993eff204ca7206cabe0fc6dfb75759994cacbf4c32c84ec5699a51c5d │  189634001 │    120 │
└────────────┴──────────────────────────────────────────────────

**结果 / Result**：全量数据中 2,717,844 个多行组合，8.4% 价格不同、0.3% 渠道不同；最大组合一天 570 件。
On the full data, 2,717,844 multi-row groups; 8.4% differ in price and 0.3% in channel; the largest group is 570 units in a day.

**决定 / Decision**：不去重，销量 = 行数；批量购买打标记而不是删除。下面用数据确定阈值。
No deduplication; units = row count. Bulk purchases are flagged, not deleted. The threshold is chosen from the data below.

In [14]:
# 一次比较 5 个阈值：VALUES 现场造一张小表，逗号是交叉连接，再按阈值分组
# pct_rows 已经是百分比（乘过 100）
# Compare 5 thresholds at once: VALUES builds a small table, the comma is a cross join,
# then group by threshold. pct_rows is already a percentage (multiplied by 100)
con.sql("""
WITH g AS (
    SELECT t_dat, customer_id, article_id, COUNT(*) AS n_rows
    FROM ods_transactions
    GROUP BY ALL
)
SELECT
    k AS threshold,
    SUM(CASE WHEN n_rows > k THEN 1 ELSE 0 END)      AS n_groups,
    SUM(CASE WHEN n_rows > k THEN n_rows ELSE 0 END) AS n_rows_affected,
    ROUND(100.0 * SUM(CASE WHEN n_rows > k THEN n_rows ELSE 0 END) / SUM(n_rows), 3) AS pct_rows
FROM g, (VALUES (3), (5), (10), (20), (50)) AS t(k)
GROUP BY k
ORDER BY k
""").show()

┌───────────┬──────────┬─────────────────┬──────────┐
│ threshold │ n_groups │ n_rows_affected │ pct_rows │
│   int32   │  int128  │     int128      │  double  │
├───────────┼──────────┼─────────────────┼──────────┤
│         3 │    93707 │          477149 │    1.501 │
│         5 │    19823 │          171406 │    0.539 │
│        10 │     3130 │           56305 │    0.177 │
│        20 │      636 │           21024 │    0.066 │
│        50 │       45 │            3945 │    0.012 │
└───────────┴──────────┴─────────────────┴──────────┘



**决定 / Decision**：阈值 = 5（超过 5 件，即 ≥ 6 件标记为 `is_bulk`）。
Threshold = 5 (more than 5 units, i.e. ≥ 6, is flagged `is_bulk`).

- 理由：同款同色的正常购买多为 1–2 件（不确定尺码），喜欢的款式可能 3–5 件；6 件以上更可能是批量采购。
  Rationale: normal purchases of one style and colour are usually 1–2 units (size uncertainty), sometimes 3–5; 6+ is more likely bulk buying.
- 影响 0.539% 的交易行，很少误伤正常买家。
  Affects 0.539% of rows and rarely flags normal buyers.
- 总销量和 KPI 保留批量；顾客分群、人货匹配、爆款预测剔除；敏感性分析用阈值 10。
  Keep bulk for totals and KPIs; exclude it for segmentation, affinity and best-seller work; sensitivity check with threshold 10.

## 4. 顾客表字段 / Customer fields
对应数据字典 §2.3 / See data dictionary §2.3

**问题 / Question**：`FN`、`Active` 的含义未说明；类别列有没有同义不同写法？年龄质量如何？
`FN` and `Active` are undocumented; do category columns have inconsistent spellings? How good is age?

In [16]:
con.sql("SELECT FN, Active, COUNT(*) AS n FROM ods_customers GROUP BY ALL ORDER BY n DESC").show()
con.sql("SELECT club_member_status, COUNT(*) AS n FROM ods_customers GROUP BY ALL ORDER BY n DESC").show()
con.sql("SELECT fashion_news_frequency, COUNT(*) AS n FROM ods_customers GROUP BY ALL ORDER BY n DESC").show()

┌────────┬────────┬────────┐
│   FN   │ Active │   n    │
│ double │ double │ int64  │
├────────┼────────┼────────┤
│   NULL │   NULL │ 895050 │
│    1.0 │    1.0 │ 464404 │
│    1.0 │   NULL │  12526 │
└────────┴────────┴────────┘

┌────────────────────┬─────────┐
│ club_member_status │    n    │
│      varchar       │  int64  │
├────────────────────┼─────────┤
│ ACTIVE             │ 1272491 │
│ PRE-CREATE         │   92960 │
│ NULL               │    6062 │
│ LEFT CLUB          │     467 │
└────────────────────┴─────────┘

┌────────────────────────┬────────┐
│ fashion_news_frequency │   n    │
│        varchar         │ int64  │
├────────────────────────┼────────┤
│ NONE                   │ 877711 │
│ Regularly              │ 477416 │
│ NULL                   │  16009 │
│ Monthly                │    842 │
│ None                   │      2 │
└────────────────────────┴────────┘



In [17]:
# 年龄：COUNT(*) FILTER (WHERE ...) 只数满足条件的行，等价于 SUM(CASE WHEN ...)
# Age: COUNT(*) FILTER (WHERE ...) counts matching rows only, same as SUM(CASE WHEN ...)
con.sql("""
SELECT
    ROUND(100.0 * COUNT(*) FILTER (WHERE age IS NULL) / COUNT(*), 2) AS pct_null,
    MIN(age)                 AS min_age,
    QUANTILE_CONT(age, 0.5)  AS median_age,
    MAX(age)                 AS max_age
FROM ods_customers
""").show()

┌──────────┬─────────┬────────────┬─────────┐
│ pct_null │ min_age │ median_age │ max_age │
│  double  │  int64  │   double   │  int64  │
├──────────┼─────────┼────────────┼─────────┤
│     1.16 │      16 │       32.0 │      99 │
└──────────┴─────────┴────────────┴─────────┘



In [18]:
# 交叉验证：FN 是否就是「是否订阅资讯」？两个方向都要看
# Cross-check: is FN the newsletter subscription flag? Check both directions
con.sql("""
SELECT FN, fashion_news_frequency, COUNT(*) AS n
FROM ods_customers
GROUP BY ALL
ORDER BY FN, n DESC
""").show()

┌────────┬────────────────────────┬────────┐
│   FN   │ fashion_news_frequency │   n    │
│ double │        varchar         │ int64  │
├────────┼────────────────────────┼────────┤
│    1.0 │ Regularly              │ 475310 │
│    1.0 │ Monthly                │    829 │
│    1.0 │ NONE                   │    789 │
│    1.0 │ None                   │      2 │
│   NULL │ NONE                   │ 876922 │
│   NULL │ NULL                   │  16009 │
│   NULL │ Regularly              │   2106 │
│   NULL │ Monthly                │     13 │
└────────┴────────────────────────┴────────┘



**结论 / Conclusion**

- FN = 1 的人中 99.83% 在订阅；FN 为 NULL 的人中 99.76% 未订阅 → FN 解释为「是否订阅资讯」，NULL 视为 0。约 0.2% 不一致，原因无法确认，保持原样。
  99.83% of FN = 1 customers subscribe and 99.76% of FN = NULL do not → FN is the subscription flag and NULL means 0. About 0.2% disagree for unknown reasons and are left as is.
- `fashion_news_frequency` 中 `None`（2 人）和 `NONE` 并存 → DWD 统一为 `NONE`。
  `None` (2 customers) coexists with `NONE` → normalised to `NONE` at DWD.
- 年龄空值 1.16%，范围 16–99，中位数 32 → 年龄分群设 UNKNOWN 组。
  Age is 1.16% null, 16–99, median 32 → age groups include UNKNOWN.

## 5. DWD 层检查 / DWD checks
对应数据字典 §6 / See data dictionary §6

建表规则在 `sql/02_dwd.sql`；跨层核对在 `checks.py`。这里看取值分布和不完整周。
Build rules are in `sql/02_dwd.sql`; cross-layer checks are in `checks.py`. Here we look at value distributions and partial weeks.

In [19]:
# 每个清洗后的列，确认没有漏掉的取值
# For each cleaned column, confirm no value slipped through
for col in ["is_fn", "is_active", "club_member_status", "fashion_news_frequency", "age_group"]:
    con.sql(f"SELECT {col}, COUNT(*) AS n FROM dwd_customers GROUP BY {col} ORDER BY n DESC").show()

┌───────┬────────┐
│ is_fn │   n    │
│ int32 │ int64  │
├───────┼────────┤
│     0 │ 895050 │
│     1 │ 476930 │
└───────┴────────┘

┌───────────┬────────┐
│ is_active │   n    │
│   int32   │ int64  │
├───────────┼────────┤
│         0 │ 907576 │
│         1 │ 464404 │
└───────────┴────────┘

┌────────────────────┬─────────┐
│ club_member_status │    n    │
│      varchar       │  int64  │
├────────────────────┼─────────┤
│ ACTIVE             │ 1272491 │
│ PRE-CREATE         │   92960 │
│ UNKNOWN            │    6062 │
│ LEFT CLUB          │     467 │
└────────────────────┴─────────┘

┌────────────────────────┬────────┐
│ fashion_news_frequency │   n    │
│        varchar         │ int64  │
├────────────────────────┼────────┤
│ NONE                   │ 877713 │
│ Regularly              │ 477416 │
│ UNKNOWN                │  16009 │
│ Monthly                │    842 │
└────────────────────────┴────────┘

┌───────────┬────────┐
│ age_group │   n    │
│  varchar  │ int64  │
├───────────

In [20]:
# 哪些周不完整？预期只有首尾两周
# Which weeks are partial? Expect only the first and last
con.sql("""
SELECT week_start, COUNT(DISTINCT t_dat) AS n_days, COUNT(*) AS n_rows
FROM dwd_transactions
WHERE is_partial_week
GROUP BY week_start
ORDER BY week_start
""").show()

┌────────────┬────────┬────────┐
│ week_start │ n_days │ n_rows │
│    date    │ int64  │ int64  │
├────────────┼────────┼────────┤
│ 2018-09-17 │      4 │ 165807 │
│ 2020-09-21 │      2 │  64996 │
└────────────┴────────┴────────┘



**结果 / Result**：只有 2018-09-17（4 天）和 2020-09-21（2 天）两周不完整，中间没有缺日期的周。
Only 2018-09-17 (4 days) and 2020-09-21 (2 days) are partial; no mid-period week is missing dates.

**为什么批量标记用窗口函数而不是 GROUP BY / Why a window function, not GROUP BY, for the bulk flag**：GROUP BY 会把表从「一件」压成「一个组合」，改变粒度，约 320 万件（约 10%）销量会消失；窗口函数保留每一行。
GROUP BY would collapse the grain from "one unit" to "one group", losing about 3.2 million units (~10%); a window function keeps every row.

## 6. DWS 层：不可加指标 / DWS: non-additive metrics
对应数据字典 §7.1、§7.2、§7.4 / See data dictionary §7.1, §7.2, §7.4

**问题 / Question**：从小类加总到大类时，动销 SKU 数和购买人数能直接相加吗？
When rolling product types up to a product group, can distinct articles and distinct buyers be summed?

**预期 / Expectation**：动销 SKU 可以（每个商品只属于一个小类，各组不重叠）；人数不行（同一个人会在多个小类购买）。
Distinct articles can (each article belongs to one type, so groups do not overlap); buyers cannot (one person buys in several types).

In [21]:
# from_dws：各小类直接相加；from_dwd：回到明细去重（真实值）
# from_dws: sum across types; from_dwd: distinct count from the detail table (true value)
con.sql("""
WITH from_dws AS (
    SELECT week_start,
           SUM(n_articles_sold) AS sum_articles,
           SUM(n_customers)     AS sum_customers
    FROM dws_week_category
    WHERE product_group_name = 'Garment Upper body'
    GROUP BY week_start
),
from_dwd AS (
    SELECT week_start,
           COUNT(DISTINCT article_id)  AS true_articles,
           COUNT(DISTINCT customer_id) AS true_customers
    FROM dwd_transactions
    WHERE product_group_name = 'Garment Upper body'
      AND NOT is_partial_week
    GROUP BY week_start
)
SELECT *
FROM from_dws JOIN from_dwd USING (week_start)
ORDER BY week_start
LIMIT 5
""").show()

┌────────────┬──────────────┬───────────────┬───────────────┬────────────────┐
│ week_start │ sum_articles │ sum_customers │ true_articles │ true_customers │
│    date    │    int128    │    int128     │     int64     │     int64      │
├────────────┼──────────────┼───────────────┼───────────────┼────────────────┤
│ 2018-09-24 │        10402 │        133131 │         10402 │          78532 │
│ 2018-10-01 │         9289 │         82086 │          9289 │          51182 │
│ 2018-10-08 │        10972 │         97055 │         10972 │          57193 │
│ 2018-10-15 │        10570 │         92112 │         10570 │          54308 │
│ 2018-10-22 │        10601 │        116140 │         10601 │          69890 │
└────────────┴──────────────┴───────────────┴───────────────┴────────────────┘



**结果 / Result**：动销 SKU 完全一致；人数直接相加高估约 60–70%。
Distinct articles match exactly; summing buyers overstates by about 60–70%.

**规律 / Rule**：各组去重数能相加，当且仅当各组之间没有重叠。去重指标必须回到明细计算。
Distinct counts add up only when the groups do not overlap. Distinct metrics must be computed from the detail table.

In [22]:
# Unknown 品类的销量占比；括号里的子查询先算出分母
# Share of units in the Unknown category; the subquery in brackets computes the denominator
con.sql("""
SELECT SUM(units) AS units,
       ROUND(100.0 * SUM(units) / (SELECT SUM(units) FROM dws_week_category), 2) AS pct
FROM dws_week_category
WHERE product_group_name = 'Unknown'
""").show()

┌────────┬────────┐
│ units  │  pct   │
│ int128 │ double │
├────────┼────────┤
│  96009 │    0.3 │
└────────┴────────┘



**结果 / Result**：全量数据中 Unknown 占 0.3%，分析中单列一组。
Unknown is 0.3% of units on the full data and is kept as its own group.

## 7. 商品生命周期与删失 / Article lifecycle and censoring
对应数据字典 §7.3 / See data dictionary §7.3

**问题 / Question**：数据没有上新日期，用首次出售日代替。但首次出售日 = 数据第一天的商品，真的是那天上新的吗？
There is no launch date, so first sale date is the proxy. But was an article first seen on the data's first day really launched that day?

In [23]:
# 每周首次出售的商品数：前几周异常偏高说明存在左删失
# Articles first sold per week: very high counts in the first weeks indicate left censoring
con.sql("""
SELECT first_sale_week, COUNT(*) AS n_new_articles
FROM dws_articles
GROUP BY first_sale_week
ORDER BY first_sale_week
LIMIT 15
""").show()

┌─────────────────┬────────────────┐
│ first_sale_week │ n_new_articles │
│      date       │     int64      │
├─────────────────┼────────────────┤
│ 2018-09-17      │          18450 │
│ 2018-09-24      │           7248 │
│ 2018-10-01      │           2213 │
│ 2018-10-08      │           3289 │
│ 2018-10-15      │           2051 │
│ 2018-10-22      │           1408 │
│ 2018-10-29      │            973 │
│ 2018-11-05      │            933 │
│ 2018-11-12      │            838 │
│ 2018-11-19      │            894 │
│ 2018-11-26      │            846 │
│ 2018-12-03      │            873 │
│ 2018-12-10      │            819 │
│ 2018-12-17      │            748 │
│ 2018-12-24      │            502 │
└─────────────────┴────────────────┘
  15 rows                2 columns



**结果 / Result**：首周 18,450 个「新品」，之后快速下降，2018-10-29 起稳定在约 800–970 个。
18,450 "new" articles in the first week, falling fast and stabilising at ~800–970 from 2018-10-29.

**决定 / Decisions**

- **左删失 / Left censoring**：首次出售早于 2018-10-29 的商品不算可观测的新品（`is_launch_observed`）；敏感性分析用 2018-12-03。
  Articles first sold before 2018-10-29 are not observed launches; sensitivity check with 2018-12-03.
- **右删失 / Right censoring**：上新周 + 7 周不晚于最后一个完整周（2020-09-14），即上新周 ≤ 2020-07-27，才参与爆款判定（`has_full_8w`）。
  Only articles whose launch week + 7 weeks is no later than the last full week (2020-09-14) enter best-seller labelling.

In [24]:
# 两个标记的组合；FALSE/FALSE 在逻辑上不可能出现
# Flag combinations; FALSE/FALSE is logically impossible
con.sql("""
SELECT is_launch_observed, has_full_8w, COUNT(*) AS n_articles
FROM dws_articles
GROUP BY ALL
ORDER BY ALL
""").show()

┌────────────────────┬─────────────┬────────────┐
│ is_launch_observed │ has_full_8w │ n_articles │
│      boolean       │   boolean   │   int64    │
├────────────────────┼─────────────┼────────────┤
│ false              │ true        │      34659 │
│ true               │ false       │       6029 │
│ true               │ true        │      63859 │
└────────────────────┴─────────────┴────────────┘



**结果 / Result**：全量数据中 63,859 个商品（61%）两个条件都满足，可用于爆款分析；34,659 个左删失，6,029 个右删失。
On the full data, 63,859 articles (61%) meet both conditions and are eligible for best-seller analysis; 34,659 are left-censored and 6,029 right-censored.

---

## 下一步 / Next steps

- 模块 2：商品结构、趋势（STL 分解）、上新节奏与爆款预判 → `notebooks/02_product_analysis.ipynb`
  Module 2: product mix, trends (STL), launch cadence and best-seller prediction
- 待确认 / Open items：渠道含义、商品层级关系、极小价格（见数据字典 §5）
  Channel meaning, article hierarchy, very small prices (see data dictionary §5)

In [ ]:
con.close()